# Car Sales — Feature Selection
**Team LGTSOW**
**Notebook 4 — Feature Selection**
**File:** `P1_CarSales-4_FeatureSelection_LGTSOW.ipynb`

> **Note on execution:** Like Notebook 3, this was authored in an environment without
> internet access and without `xgboost`/`tensorflow`/`statsmodels` installed, so it
> could not be executed here. Run it top-to-bottom in the team's conda environment
> before committing. `statsmodels` (for VIF) is an additional dependency beyond what
> Notebook 3 needed -- if it isn't already in the course environment, install it the
> same way `xgboost` was installed, and remember to re-export `environment.yml`
> afterward.

> **Portability:** Same approach as Notebook 3 -- ACES `sys.path` fix, a portable
> `DATA_DIR` resolver, and SLURM-aware `GridSearchCV` parallelism sizing, so this
> file runs unmodified on either a laptop or ACES.

This notebook trims the feature set two ways -- Random Forest importance, then VIF --
and retrains every model from Notebook 3 on the reduced features to see whether
performance holds up, and how much faster/slower training gets.

In [1]:
import sys, os

pkg_dir = os.path.expandvars('$SCRATCH/python_packages')
if os.path.isdir(pkg_dir) and pkg_dir not in sys.path:
    sys.path.insert(0, pkg_dir)
    print(f"Added to sys.path: {pkg_dir}")

Added to sys.path: /scratch/user/u.gd352312/python_packages


In [2]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

from xgboost import XGBRegressor

from statsmodels.stats.outliers_influence import variance_inflation_factor

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

I0000 00:00:1789601891.910279 3422629 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789601892.213596 3422629 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789601913.486693 3422629 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789601929.218419 3422629 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them 

## 1. Setup

### Data location and Notebook 3 baseline

Same portable data-directory resolution as Notebook 3. We also read in Notebook 3's
`model_metrics.csv` here -- per the assignment, we need it later for the
before/after comparison, but we do **not** reload the saved `.pkl`/`.keras` models
themselves; everything gets retrained from scratch on the reduced feature set.

In [3]:
# Checked in order: an explicit override via the CARSALES_DATA_DIR environment
# variable, the notebook's own working directory, then a couple of common ACES
# scratch/project directory conventions. Add your own path here if needed.
CANDIDATE_DATA_DIRS = [
    os.environ.get('CARSALES_DATA_DIR'),
    '.',
    os.path.expanduser('~/scratch/msds565'),
    os.path.expanduser('~/msds565/data'),
]

DATA_DIR = None
for candidate in CANDIDATE_DATA_DIRS:
    if candidate and os.path.exists(os.path.join(candidate, 'processed_train.csv')):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find processed_train.csv in any candidate data directory: "
        f"{[c for c in CANDIDATE_DATA_DIRS if c]}\n"
        "Set the CARSALES_DATA_DIR environment variable to your class data folder."
    )

print(f"Using data directory: {os.path.abspath(DATA_DIR)}")

Using data directory: /scratch/user/u.gd352312/Car_Sales


In [4]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'processed_train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'processed_test.csv'))

print(f"Train: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns")
print(f"Test:  {test_df.shape[0]:,} rows x {test_df.shape[1]} columns")

Train: 241,847 rows x 278 columns
Test:  60,462 rows x 278 columns


**Building X and y -- identical approach to Notebook 3.** `price` is the
target; every object-dtype column (the `meta_`-prefixed fairness-audit columns, plus
`listed_date`) is dropped from `X`, not just the ones with the `meta_` prefix --
detecting by dtype is robust regardless of naming convention.

In [5]:
metadata_cols = train_df.select_dtypes(include='object').columns.tolist()
print(f"Dropping {len(metadata_cols)} object-dtype column(s) from X: {metadata_cols}")

y_train = train_df['price']
X_train = train_df.drop(columns=['price'] + metadata_cols)

y_test = test_df['price']
X_test = test_df.drop(columns=['price'] + metadata_cols)

print(f"X_train: {X_train.shape},  X_test: {X_test.shape}")

Dropping 16 object-dtype column(s) from X: ['meta_city', 'meta_bed', 'meta_body_type', 'meta_cabin', 'meta_exterior_color', 'meta_franchise_make', 'meta_fuel_type', 'meta_interior_color', 'listed_date', 'meta_listing_color', 'meta_make_name', 'meta_model_name', 'meta_transmission', 'meta_trim_name', 'meta_wheel_system', 'meta_engine_layout']
X_train: (241847, 261),  X_test: (60462, 261)


In [6]:
baseline_metrics = pd.read_csv(os.path.join(DATA_DIR, 'model_metrics.csv'), index_col=0)
print("Notebook 3 baseline performance:")
baseline_metrics

Notebook 3 baseline performance:


,cv_MAE_mean,cv_MAPE_mean,cv_R2_mean,test_MAE,test_MAPE,test_R2,training_time_sec
model,,,,,,,
linreg,4582.383364,0.227486,0.843945,4597.093078,0.228420,0.841928,2.799440
rf,1542.156152,0.068014,0.971051,1470.479823,0.065453,0.973958,1572.733973
xgb,1605.542833,0.068122,0.973958,1578.857217,0.067062,0.976568,24.620455
nn3,2041.481006,0.084218,0.961962,1962.663824,0.080302,0.964234,63.370022


**Sizing `GridSearchCV` parallelism.** Same SLURM-aware approach as Notebook 3 --
prefer the job's actual allocation (`SLURM_CPUS_PER_TASK`, `SLURM_MEM_PER_NODE`/
`SLURM_MEM_PER_CPU`) over host-wide `os.cpu_count()`/`psutil` numbers, which can
wildly overstate what a shared HPC node's job actually has available.

In [7]:
try:
    import psutil
    available_bytes = psutil.virtual_memory().available
except ImportError:
    available_bytes = None

slurm_cpus = os.environ.get('SLURM_CPUS_PER_TASK') or os.environ.get('SLURM_JOB_CPUS_PER_NODE')
if slurm_cpus is not None:
    cpu_count = int(slurm_cpus.split(',')[0])
    cpu_source = 'SLURM allocation'
else:
    cpu_count = os.cpu_count() or 1
    cpu_source = 'os.cpu_count() (whole machine -- not SLURM-aware)'

bytes_per_job = X_train.memory_usage(deep=True).sum()

slurm_mem_mb = os.environ.get('SLURM_MEM_PER_NODE') or os.environ.get('SLURM_MEM_PER_CPU')
if slurm_mem_mb is not None:
    per_cpu = 'SLURM_MEM_PER_CPU' in os.environ
    available_bytes = int(slurm_mem_mb) * 1e6 * (cpu_count if per_cpu else 1)
    mem_source = 'SLURM allocation'
elif available_bytes is not None:
    mem_source = "psutil (whole machine -- not SLURM-aware)"
else:
    mem_source = 'unavailable'

HARD_CAP = 32  # adjust to your allocated core count once memory is confirmed sufficient

if available_bytes is not None:
    SAFETY_FRACTION = 0.5
    max_jobs_by_memory = max(1, int((available_bytes * SAFETY_FRACTION) // bytes_per_job))
else:
    max_jobs_by_memory = 2

N_JOBS = max(1, min(cpu_count, max_jobs_by_memory, HARD_CAP))

print(f"CPU count source: {cpu_source}  ({cpu_count} CPUs)")
print(f"Memory source:     {mem_source}")
print(f"--> Using N_JOBS = {N_JOBS} for GridSearchCV")

CPU count source: SLURM allocation  (32 CPUs)
Memory source:     SLURM allocation
--> Using N_JOBS = 32 for GridSearchCV


## 2. Importance-Based Filtering

We fit a single Random Forest on the full `X_train` (261 features) purely to rank
feature importance -- this is not a tuned model and isn't exported; `GridSearchCV`
already tunes and exports the real Random Forest in Section 4. A generous
`n_estimators` gives a stable importance ranking without needing to search
hyperparameters for this purpose.

In [8]:
importance_rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=N_JOBS)
importance_rf.fit(X_train, y_train)

importances = pd.Series(importance_rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)

TOP_N = 100
top_features = importances.head(TOP_N).index.tolist()

print(f"Retaining the top {TOP_N} of {X_train.shape[1]} features by Random Forest importance.")
print()
print("Top 15 features by importance:")
print(importances.head(15).to_string())

Retaining the top 100 of 261 features by Random Forest importance.

Top 15 features by importance:
horsepower               0.437011
mileage                  0.262746
is_luxury_make           0.073777
torque_lbft              0.030118
year                     0.029080
height                   0.015061
opt_sunroof_moonroof     0.013093
opt_navigation_system    0.008366
fuel_tank_volume         0.008344
is_new                   0.007922
width                    0.006007
wheelbase                0.005331
back_legroom             0.005066
torque_rpm               0.004515
vehicle_age              0.004458


**Interpretation:** the top of this ranking should be dominated by the
continuous specs the EDA and modeling notebooks already flagged as strongly
price-correlated (horsepower, engine size, vehicle age/year, mileage), plus whichever
one-hot dummies for high-value makes/body-types carry real signal. Features near the
bottom of the full 261 -- rare one-hot categories from the "long tail beyond top 20"
truncation in Notebook 2, or option flags most listings share -- are the ones we'd
expect to get cut here.

In [9]:
X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

print(f"X_train_top: {X_train_top.shape}")
print(f"X_test_top:  {X_test_top.shape}")

X_train_top: (241847, 100)
X_test_top:  (60462, 100)


## 3. VIF Filtering

Variance Inflation Factor measures how much a feature's variance is inflated by its
linear relationship with the *other* features currently in the set -- VIF > 10 is the
conventional "this feature is redundant given what's already here" threshold. We
already found real multicollinearity in the EDA notebook (horsepower vs. the hp
parsed from `power`, city vs. highway fuel economy, engine size vs. vehicle
dimensions) — VIF filtering is where that finally gets acted on.

**Iterative, one column at a time:** dropping every high-VIF column in a single pass
is wrong, because removing one collinear column can change every other column's VIF
(the redundancy was relative to the whole set, not fixed per-column). So each round
we compute VIF for every remaining feature, drop only the single worst offender if
it's above 10, and repeat until nothing exceeds the threshold.

In [10]:
def iterative_vif_filter(X, threshold=10.0):
    """Repeatedly drop the single highest-VIF column until every remaining
    column's VIF is at or below `threshold`. Returns the reduced DataFrame."""
    X_reduced = X.copy()

    # A zero-variance column produces an undefined (NaN/inf) VIF and would break
    # the loop -- drop any beforehand rather than let VIF crash on them.
    zero_var_cols = X_reduced.columns[X_reduced.std() == 0].tolist()
    if zero_var_cols:
        print(f"Dropping {len(zero_var_cols)} zero-variance column(s) before VIF: {zero_var_cols}")
        X_reduced = X_reduced.drop(columns=zero_var_cols)

    round_num = 0
    while X_reduced.shape[1] > 1:
        round_num += 1
        vif_values = pd.Series(
            [variance_inflation_factor(X_reduced.values, i) for i in range(X_reduced.shape[1])],
            index=X_reduced.columns,
        )
        max_vif = vif_values.max()
        if max_vif <= threshold:
            print(f"Round {round_num}: max VIF = {max_vif:.1f} -- at or below threshold, stopping.")
            break
        worst_col = vif_values.idxmax()
        print(f"Round {round_num}: dropping '{worst_col}' (VIF = {max_vif:.1f})")
        X_reduced = X_reduced.drop(columns=[worst_col])

    return X_reduced


X_train_selected = iterative_vif_filter(X_train_top, threshold=10.0)
selected_features = X_train_selected.columns.tolist()
X_test_selected = X_test_top[selected_features]

print()
print(f"Features after importance filtering: {X_train_top.shape[1]}")
print(f"Features after VIF filtering:         {X_train_selected.shape[1]}")

/sw/eb/sw/Anaconda3/2024.02-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:198: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


Round 1: dropping 'horsepower_missing' (VIF = inf)


/sw/eb/sw/Anaconda3/2024.02-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:198: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


Round 2: dropping 'torque_rpm_missing' (VIF = inf)
Round 3: dropping 'year' (VIF = 260.6)
Round 4: dropping 'wheelbase' (VIF = 34.0)
Round 5: dropping 'city_fuel_economy' (VIF = 23.0)
Round 6: dropping 'is_new' (VIF = 20.6)
Round 7: dropping 'length' (VIF = 15.6)
Round 8: dropping 'cylinder_count' (VIF = 14.6)
Round 9: dropping 'n_major_options' (VIF = 13.1)
Round 10: dropping 'horsepower' (VIF = 12.5)
Round 11: dropping 'power_rpm_missing' (VIF = 11.5)
Round 12: max VIF = 8.2 -- at or below threshold, stopping.

Features after importance filtering: 100
Features after VIF filtering:         89


**Interpretation:** the columns VIF removes here should line up with the
redundant pairs already identified in the EDA notebook's multicollinearity heatmap
-- if, say, both `horsepower` and the one-hot dummies correlated with vehicle size
survive together, VIF is confirming they still carry distinct information even after
importance filtering; if one consistently gets dropped, that confirms it was
genuinely redundant given the others.

## 4. Retrain on the Selected Feature Set

**Same models, same parameter grids, same procedure as Notebook 3** -- Linear
Regression, Random Forest, and XGBoost via 5-fold `GridSearchCV`, plus the same 3
Keras architectures via manual 5-fold CV -- just fit against `X_train_selected`
(VIF-and-importance-reduced) instead of the full 261-feature `X_train`. Reusing the
exact grids (rather than re-tuning them) isolates the effect of feature selection
itself, which is the comparison Section 5 needs to make.

In [11]:
def run_gridsearch(name, estimator, param_grid, X_train, y_train, X_test, y_test, n_jobs=N_JOBS):
    """Run 5-fold GridSearchCV, then report CV-mean and held-out test metrics
    for the best (refit) estimator, plus its refit training time. Identical to
    Notebook 3's helper -- kept here so this notebook is self-contained."""
    scoring = {
        'MAE': 'neg_mean_absolute_error',
        'MAPE': 'neg_mean_absolute_percentage_error',
        'R2': 'r2',
    }

    gs = GridSearchCV(
        estimator, param_grid,
        cv=5, scoring=scoring, refit='R2', n_jobs=n_jobs, pre_dispatch='n_jobs'
    )
    gs.fit(X_train, y_train)

    best_idx = gs.best_index_
    cv_MAE_mean = -gs.cv_results_['mean_test_MAE'][best_idx]
    cv_MAPE_mean = -gs.cv_results_['mean_test_MAPE'][best_idx]
    cv_R2_mean = gs.cv_results_['mean_test_R2'][best_idx]

    per_fold_R2 = [gs.cv_results_[f'split{i}_test_R2'][best_idx] for i in range(gs.n_splits_)]
    best_fold = int(np.argmax(per_fold_R2)) + 1

    test_preds = gs.best_estimator_.predict(X_test)
    test_MAE = mean_absolute_error(y_test, test_preds)
    test_MAPE = mean_absolute_percentage_error(y_test, test_preds)
    test_R2 = r2_score(y_test, test_preds)

    print(f"[{name}] best params: {gs.best_params_}")
    print(f"[{name}] CV  -> MAE={cv_MAE_mean:,.1f}  MAPE={cv_MAPE_mean:.3f}  R2={cv_R2_mean:.3f}")
    print(f"[{name}] Test-> MAE={test_MAE:,.1f}  MAPE={test_MAPE:.3f}  R2={test_R2:.3f}")
    print(f"[{name}] refit training time: {gs.refit_time_:.1f}s")
    print(f"[{name}] best-scoring fold: {best_fold} (of 5)")

    row = {
        'model': name,
        'cv_MAE_mean': cv_MAE_mean,
        'cv_MAPE_mean': cv_MAPE_mean,
        'cv_R2_mean': cv_R2_mean,
        'test_MAE': test_MAE,
        'test_MAPE': test_MAPE,
        'test_R2': test_R2,
        'training_time_sec': gs.refit_time_,
        'best_fold': best_fold,
    }
    return row, gs

In [12]:
linreg_grid = {
    'fit_intercept': [True, False],
    'positive': [False, True],
}
linreg_row, linreg_gs = run_gridsearch(
    'linreg', LinearRegression(), linreg_grid,
    X_train_selected, y_train, X_test_selected, y_test
)

[linreg] best params: {'fit_intercept': True, 'positive': False}
[linreg] CV  -> MAE=5,333.2  MAPE=0.266  R2=0.784
[linreg] Test-> MAE=5,349.7  MAPE=0.267  R2=0.782
[linreg] refit training time: 0.7s
[linreg] best-scoring fold: 3 (of 5)


In [13]:
rf_grid = {
    'n_estimators': [100, 300],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 4],
}
rf_row, rf_gs = run_gridsearch(
    'rf', RandomForestRegressor(random_state=42), rf_grid,
    X_train_selected, y_train, X_test_selected, y_test
)

/sw/eb/sw/Anaconda3/2024.02-1/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[rf] best params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 300}
[rf] CV  -> MAE=1,602.5  MAPE=0.071  R2=0.969
[rf] Test-> MAE=1,526.7  MAPE=0.068  R2=0.972
[rf] refit training time: 847.8s
[rf] best-scoring fold: 3 (of 5)


In [14]:
xgb_grid = {
    'n_estimators': [100, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
}
xgb_row, xgb_gs = run_gridsearch(
    'xgb', XGBRegressor(random_state=42, objective='reg:squarederror'), xgb_grid,
    X_train_selected, y_train, X_test_selected, y_test
)

[xgb] best params: {'learning_rate': 0.2, 'max_depth': 8, 'n_estimators': 300}
[xgb] CV  -> MAE=1,639.9  MAPE=0.069  R2=0.973
[xgb] Test-> MAE=1,612.6  MAPE=0.068  R2=0.976
[xgb] refit training time: 12.0s
[xgb] best-scoring fold: 3 (of 5)


**Keras architectures -- identical definitions to Notebook 3.** Same three
candidates (small, medium-regularized, wide-batchnorm), re-run through manual 5-fold
CV on the reduced feature set. The winning architecture isn't necessarily the same
one that won in Notebook 3 -- fewer, less collinear input features can change which
architecture generalizes best.

In [15]:
def build_nn1(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


def build_nn2(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


def build_nn3(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


architectures = {'nn1': build_nn1, 'nn2': build_nn2, 'nn3': build_nn3}

In [16]:
X_train_arr = X_train_selected.to_numpy()
y_train_arr = y_train.to_numpy()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

keras_cv_results = {}
keras_fold_r2_by_arch = {}

for name, build_fn in architectures.items():
    fold_mae, fold_mape, fold_r2 = [], [], []

    for fold_i, (tr_idx, val_idx) in enumerate(kf.split(X_train_arr), start=1):
        X_tr, X_val = X_train_arr[tr_idx], X_train_arr[val_idx]
        y_tr, y_val = y_train_arr[tr_idx], y_train_arr[val_idx]

        model = build_fn(X_train_arr.shape[1])
        early_stop = keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True
        )
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=50, batch_size=256,
            callbacks=[early_stop], verbose=0,
        )

        preds = model.predict(X_val, verbose=0).flatten()
        fold_mae.append(mean_absolute_error(y_val, preds))
        fold_mape.append(mean_absolute_percentage_error(y_val, preds))
        fold_r2.append(r2_score(y_val, preds))

        print(f"[{name}] fold {fold_i}: MAE={fold_mae[-1]:,.1f}  MAPE={fold_mape[-1]:.3f}  R2={fold_r2[-1]:.3f}")

    keras_cv_results[name] = {
        'cv_MAE_mean': np.mean(fold_mae),
        'cv_MAPE_mean': np.mean(fold_mape),
        'cv_R2_mean': np.mean(fold_r2),
        'cv_R2_std': np.std(fold_r2),
    }
    keras_fold_r2_by_arch[name] = fold_r2
    print(f"[{name}] CV mean R2 = {keras_cv_results[name]['cv_R2_mean']:.3f}  (std {keras_cv_results[name]['cv_R2_std']:.3f})")
    print()

keras_cv_summary = pd.DataFrame(keras_cv_results).T
keras_cv_summary

E0000 00:00:1789605861.090687 3422629 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1789605861.993409 3424363 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
E0000 00:00:1789605862.676473 3422629 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1789605862.676633 3424363 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1789605862.696618 3422629 gpu_device.cc:2365] Cannot dlopen some GPU l

[nn1] fold 1: MAE=2,801.4  MAPE=0.115  R2=0.933
[nn1] fold 2: MAE=2,900.3  MAPE=0.119  R2=0.929
[nn1] fold 3: MAE=2,794.9  MAPE=0.114  R2=0.930
[nn1] fold 4: MAE=2,878.1  MAPE=0.116  R2=0.931
[nn1] fold 5: MAE=2,845.4  MAPE=0.115  R2=0.928
[nn1] CV mean R2 = 0.930  (std 0.002)

[nn2] fold 1: MAE=2,660.0  MAPE=0.118  R2=0.940
[nn2] fold 2: MAE=2,674.4  MAPE=0.117  R2=0.940
[nn2] fold 3: MAE=2,638.3  MAPE=0.112  R2=0.937
[nn2] fold 4: MAE=2,661.1  MAPE=0.116  R2=0.940
[nn2] fold 5: MAE=2,675.4  MAPE=0.110  R2=0.937
[nn2] CV mean R2 = 0.939  (std 0.002)

[nn3] fold 1: MAE=2,238.8  MAPE=0.095  R2=0.958
[nn3] fold 2: MAE=2,286.5  MAPE=0.095  R2=0.956
[nn3] fold 3: MAE=2,238.5  MAPE=0.093  R2=0.954
[nn3] fold 4: MAE=2,230.4  MAPE=0.091  R2=0.956
[nn3] fold 5: MAE=2,197.3  MAPE=0.090  R2=0.959
[nn3] CV mean R2 = 0.957  (std 0.002)



,cv_MAE_mean,cv_MAPE_mean,cv_R2_mean,cv_R2_std
nn1,2844.007355,0.115787,0.930335,0.001657
nn2,2661.811686,0.114744,0.938606,0.001541
nn3,2238.305368,0.092871,0.956758,0.001703


In [17]:
best_arch_name = keras_cv_summary['cv_R2_mean'].idxmax()
print(f"Best architecture on the selected feature set: {best_arch_name}")

best_build_fn = architectures[best_arch_name]

start_time = time.time()
final_nn = best_build_fn(X_train_arr.shape[1])
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)
final_nn.fit(
    X_train_arr, y_train_arr,
    validation_split=0.1,
    epochs=50, batch_size=256,
    callbacks=[early_stop], verbose=0,
)
nn_training_time = time.time() - start_time

nn_test_preds = final_nn.predict(X_test_selected.to_numpy(), verbose=0).flatten()
nn_test_MAE = mean_absolute_error(y_test, nn_test_preds)
nn_test_MAPE = mean_absolute_percentage_error(y_test, nn_test_preds)
nn_test_R2 = r2_score(y_test, nn_test_preds)

nn_best_fold = int(np.argmax(keras_fold_r2_by_arch[best_arch_name])) + 1

nn_row = {
    'model': best_arch_name,
    'cv_MAE_mean': keras_cv_summary.loc[best_arch_name, 'cv_MAE_mean'],
    'cv_MAPE_mean': keras_cv_summary.loc[best_arch_name, 'cv_MAPE_mean'],
    'cv_R2_mean': keras_cv_summary.loc[best_arch_name, 'cv_R2_mean'],
    'test_MAE': nn_test_MAE,
    'test_MAPE': nn_test_MAPE,
    'test_R2': nn_test_R2,
    'training_time_sec': nn_training_time,
    'best_fold': nn_best_fold,
}

print(f"[{best_arch_name}] Test -> MAE={nn_test_MAE:,.1f}  MAPE={nn_test_MAPE:.3f}  R2={nn_test_R2:.3f}")
print(f"[{best_arch_name}] refit training time: {nn_training_time:.1f}s")

Best architecture on the selected feature set: nn3
[nn3] Test -> MAE=2,230.9  MAPE=0.093  R2=0.958
[nn3] refit training time: 46.6s


## 5. Before/After Comparison

Same table shape as Notebook 3's `model_metrics.csv`, built the same way, so the two
can be compared row-by-row.

In [18]:
performance_rows = [linreg_row, rf_row, xgb_row, nn_row]

selected_metrics = pd.DataFrame(performance_rows).set_index('model')
selected_metrics = selected_metrics[[
    'cv_MAE_mean', 'cv_MAPE_mean', 'cv_R2_mean',
    'test_MAE', 'test_MAPE', 'test_R2',
    'training_time_sec',
]]
selected_metrics

,cv_MAE_mean,cv_MAPE_mean,cv_R2_mean,test_MAE,test_MAPE,test_R2,training_time_sec
model,,,,,,,
linreg,5333.190911,0.266086,0.783808,5349.681331,0.267023,0.782265,0.729841
rf,1602.492058,0.070939,0.968903,1526.734093,0.067757,0.972325,847.756679
xgb,1639.933940,0.069294,0.972808,1612.607448,0.068448,0.975564,12.013965
nn3,2238.305368,0.092871,0.956758,2230.901008,0.092662,0.957556,46.626299


**Row-by-row comparison against Notebook 3.** Positive `test_R2` deltas and
negative `training_time` deltas are both "improved"; note that Notebook 3's winning
Keras architecture and this notebook's may have different names (`nn1`/`nn2`/`nn3`),
so we compare Keras rows by position (both notebooks' single winning network) rather
than by matching label.

In [19]:
# Notebook 3 indexes its winning keras row by whichever architecture won there
# (nn1/nn2/nn3), which may differ from the winner here -- align by role (the one
# keras row in each table) rather than by exact label match.
baseline_aligned = baseline_metrics.copy()
nn3_label = [i for i in baseline_aligned.index if i not in ('linreg', 'rf', 'xgb')][0]
baseline_aligned = baseline_aligned.rename(index={nn3_label: 'nn_winner'})

selected_aligned = selected_metrics.copy()
selected_aligned = selected_aligned.rename(index={best_arch_name: 'nn_winner'})

comparison = pd.DataFrame({
    'test_R2_before': baseline_aligned['test_R2'],
    'test_R2_after': selected_aligned['test_R2'],
    'test_R2_delta': selected_aligned['test_R2'] - baseline_aligned['test_R2'],
    'training_time_before_sec': baseline_aligned['training_time_sec'],
    'training_time_after_sec': selected_aligned['training_time_sec'],
    'training_time_delta_sec': selected_aligned['training_time_sec'] - baseline_aligned['training_time_sec'],
})
comparison['training_time_pct_change'] = (
    comparison['training_time_delta_sec'] / comparison['training_time_before_sec'] * 100
)

print(f"Feature count: {X_train.shape[1]} (before) -> {X_train_selected.shape[1]} (after), "
      f"a {(1 - X_train_selected.shape[1] / X_train.shape[1]) * 100:.0f}% reduction")
print()
comparison

Feature count: 261 (before) -> 89 (after), a 66% reduction



,test_R2_before,test_R2_after,test_R2_delta,training_time_before_sec,training_time_after_sec,training_time_delta_sec,training_time_pct_change
model,,,,,,,
linreg,0.841928,0.782265,-0.059663,2.799440,0.729841,-2.069598,-73.929024
rf,0.973958,0.972325,-0.001633,1572.733973,847.756679,-724.977294,-46.096626
xgb,0.976568,0.975564,-0.001003,24.620455,12.013965,-12.606490,-51.203320
nn_winner,0.964234,0.957556,-0.006677,63.370022,46.626299,-16.743723,-26.422151


**Fill in after running:**

- **Which models improved, which got worse, and by how much:** `[read directly off
  the test_R2_delta and training_time_pct_change columns above -- e.g. "rf's test R2
  moved by X, xgb's training time changed by Y%"]`.
- **Speed vs. accuracy trade-off:** feature selection cut the feature count from
  `[X_train.shape[1]]` to `[X_train_selected.shape[1]]`
  (`[pct reduction]`%). `[Discuss whether the training-time savings were roughly
  proportional to the feature reduction, disproportionately large (common for
  Random Forest and XGBoost, since fewer features means less work per tree split),
  or negligible for a given model type (Linear Regression's cost barely depends on
  feature count at this scale, so expect little change there).]`
- **What changed, what didn't, and why:** `[e.g. if test R2 held steady or improved
  despite dropping ~60% of features, that's evidence most of the dropped features
  were genuinely redundant (confirming the VIF/importance analysis was sound) rather
  than informative; if R2 dropped noticeably for one model in particular, discuss
  whether that model type (e.g. Random Forest, which can exploit even weakly
  informative features that VIF/importance filtering removed) was more sensitive to
  the feature cut than the others.]`

## 6. Export

**Datasets.** Unlike Notebook 3's exports, these must include `price` and the
`meta_`-prefixed columns alongside the selected features -- Notebook 5's fairness
audit and SHAP analysis need both, and dropping them here would make this the wrong
file to load there.

**Models.** Same naming convention as Notebook 3 (`P1_N<fold>_<modeltype>...`), with
the `_selected` suffix the assignment specifies for this notebook's exports.

In [20]:
train_selected_export = pd.concat(
    [train_df[['price'] + metadata_cols], X_train_selected], axis=1
)
test_selected_export = pd.concat(
    [test_df[['price'] + metadata_cols], X_test_selected], axis=1
)

train_path = os.path.join(DATA_DIR, 'selected_train.csv')
test_path = os.path.join(DATA_DIR, 'selected_test.csv')
train_selected_export.to_csv(train_path, index=False)
test_selected_export.to_csv(test_path, index=False)
print(f"Saved: {train_path}  ({train_selected_export.shape})")
print(f"Saved: {test_path}  ({test_selected_export.shape})")

metrics_path = os.path.join(DATA_DIR, 'model_metrics_selected.csv')
selected_metrics.to_csv(metrics_path)
print(f"Saved: {metrics_path}")

Saved: ./selected_train.csv  ((241847, 106))
Saved: ./selected_test.csv  ((60462, 106))
Saved: ./model_metrics_selected.csv


In [21]:
import pickle

MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

sklearn_exports = {
    'linreg': (linreg_gs.best_estimator_, linreg_row['best_fold']),
    'rf': (rf_gs.best_estimator_, rf_row['best_fold']),
    'xgb': (xgb_gs.best_estimator_, xgb_row['best_fold']),
}

for model_type, (estimator, best_fold) in sklearn_exports.items():
    filename = f"P1_N{best_fold}_{model_type}_selected.pkl"
    filepath = os.path.join(MODELS_DIR, filename)
    with open(filepath, 'wb') as f:
        pickle.dump(estimator, f)
    print(f"Saved: {filepath}")

keras_filename = f"P1_N{nn_best_fold}_{best_arch_name}_selected.keras"
keras_filepath = os.path.join(MODELS_DIR, keras_filename)
final_nn.save(keras_filepath)
print(f"Saved: {keras_filepath}")

Saved: models/P1_N3_linreg_selected.pkl
Saved: models/P1_N3_rf_selected.pkl
Saved: models/P1_N3_xgb_selected.pkl
Saved: models/P1_N5_nn3_selected.keras
